<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Device Specifications

| Specification         | Details                                      |
|-----------------------|----------------------------------------------|
| **Platform**          | Google Colaboratory                          |
| **Runtime Type**      | GPU                                          |
| **Accelerator**       | NVIDIA Tesla T4                              |
| **GPU Memory**        | 15 GB GDDR6                                  |
| **CUDA Cores**        | 2,560                                        |
| **Tensor Cores**      | 320 (2nd Gen)                                |
| **GPU Architecture**  | Turing (SM 7.5)                              |
| **CUDA Version**      | 12.2                                         |
| **CPU**               | Intel Xeon (2 vCPUs)                         |
| **System RAM**        | ~12.7 GB                                     |
| **Disk Space**        | ~78 GB                                       |
| **Python Version**    | 3.10.x                                       |
| **OS**                | Ubuntu 22.04 LTS (64-bit)                    |
| **Driver Version**    | 525.xx (NVIDIA)                              |

In [1]:
!pip install -q transformers datasets evaluate accelerate pynvml scikit-learn scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00


In [2]:
import time, gc, warnings, threading, tempfile, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

MODELS = {
    'TinyBERT':   'huawei-noah/TinyBERT_General_4L_312D',
    'DistilBERT': 'distilbert-base-uncased',
    'AlBERT':     'albert-base-v2',
    'MobileBERT': 'google/mobilebert-uncased',
    'BERT-base':  'bert-base-uncased',
}

DATASETS = {
    'SST2': ('stanfordnlp/sst2',  None,   'validation',         'sentence',               'label'),
    'QNLI': ('nyu-mll/glue',      'qnli', 'validation',         ('question','sentence'),   'label'),
    'MNLI': ('nyu-mll/glue',      'mnli', 'validation_matched', ('premise','hypothesis'),  'label'),
    'QQP':  ('nyu-mll/glue',      'qqp',  'validation',         ('question1','question2'), 'label'),
    'RTE':  ('nyu-mll/glue',      'rte',  'validation',         ('sentence1','sentence2'), 'label'),
    'CoLA': ('nyu-mll/glue',      'cola', 'validation',         'sentence',               'label'),
    'MRPC': ('nyu-mll/glue',      'mrpc', 'validation',         ('sentence1','sentence2'), 'label'),
    'STSB': ('nyu-mll/glue',      'stsb', 'validation',         ('sentence1','sentence2'), 'label'),
    'WNLI': ('nyu-mll/glue',      'wnli', 'validation',         ('sentence1','sentence2'), 'label'),
}

NUM_LABELS = {
    'SST2': 2, 'QNLI': 2, 'MNLI': 3, 'QQP': 2, 'RTE': 2,
    'CoLA': 2, 'MRPC': 2, 'STSB': 1, 'WNLI': 2
}

BATCH_SIZE       = 32
MAX_SAMPLES      = 500
MAX_LENGTH       = 128
SPARSITY         = 0.30   # 30 % of attention heads pruned  (FASP alpha)
GAMMA            = 0.50   # fraction of heads protected for performance
POLL_INTERVAL_S  = 0.005  # 5 ms GPU power polling

Device: cuda


In [3]:
try:
    import pynvml
    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    NVML_AVAILABLE = True
    print(f'pynvml ready | GPU: {pynvml.nvmlDeviceGetName(_nvml_handle)}')
except Exception as e:
    NVML_AVAILABLE = False
    print(f'pynvml unavailable ({e}) — energy will be reported as 0.0 mJ')


class PowerSampler:
    def __init__(self):
        self._samples = []
        self._running = False
        self._thread  = None

    def _poll(self):
        while self._running:
            if NVML_AVAILABLE:
                try:
                    self._samples.append(pynvml.nvmlDeviceGetPowerUsage(_nvml_handle))
                except Exception:
                    pass
            time.sleep(POLL_INTERVAL_S)

    def start(self):
        self._samples = []
        self._running = True
        self._thread  = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        self._thread.join(timeout=0.5)
        return float(np.mean(self._samples)) if self._samples else 0.0

pynvml ready | GPU: Tesla T4


# FASP — Fairness-Aware Structured Pruning

FASP prunes attention heads in two steps:
1. **Performance protection** — the top-γ heads by PPL contribution are locked (never pruned).
2. **Bias-ranked pruning** — among the remaining heads, those with the highest bias contribution are pruned first.

Here we adapt FASP for BERT-family encoder models on GLUE classification tasks.
Bias contribution is approximated by the **L2 norm of the attention-output projection weights** per head
(higher norm → head encodes stronger, potentially biased features).
Performance contribution is approximated by the **gradient magnitude** of each head's output.

In [4]:
def get_attention_heads(model):
    heads = []
    for layer_idx, layer in enumerate(model.bert.encoder.layer
                                      if hasattr(model, 'bert')
                                      else model.albert.encoder.albert_layer_groups[0].albert_layers
                                      if hasattr(model, 'albert')
                                      else model.distilbert.transformer.layer
                                      if hasattr(model, 'distilbert')
                                      else model.mobilebert.encoder.layer):
        try:
            attn = (layer.attention.self
                    if hasattr(layer, 'attention') and hasattr(layer.attention, 'self')
                    else layer.attention)
            num_heads = attn.num_attention_heads
            head_dim  = attn.attention_head_size if hasattr(attn, 'attention_head_size') \
                        else attn.query.weight.shape[0] // num_heads
            for h in range(num_heads):
                heads.append((layer_idx, h, head_dim))
        except Exception:
            pass
    return heads


def compute_head_scores(model, tok, sample_texts, device):
    model.train()
    enc = tok(sample_texts, truncation=True, padding=True,
               max_length=MAX_LENGTH, return_tensors='pt').to(device)
    dummy_labels = torch.zeros(enc['input_ids'].shape[0], dtype=torch.long).to(device)
    out = model(**enc, labels=dummy_labels)
    out.loss.backward()

    bias_scores, perf_scores = [], []

    def _collect(layer, layer_idx):
        try:
            attn = (layer.attention.self
                    if hasattr(layer, 'attention') and hasattr(layer.attention, 'self')
                    else layer.attention)
            num_heads = attn.num_attention_heads
            head_dim  = attn.attention_head_size if hasattr(attn, 'attention_head_size') \
                        else attn.query.weight.shape[0] // num_heads
            W_v = attn.value.weight
            for h in range(num_heads):
                s, e = h * head_dim, (h + 1) * head_dim
                bias_scores.append(W_v[s:e, :].norm().item())
                if W_v.grad is not None:
                    perf_scores.append(W_v.grad[s:e, :].norm().item())
                else:
                    perf_scores.append(0.0)
        except Exception:
            pass

    encoder_layers = (model.bert.encoder.layer if hasattr(model, 'bert')
                      else model.distilbert.transformer.layer if hasattr(model, 'distilbert')
                      else model.mobilebert.encoder.layer if hasattr(model, 'mobilebert')
                      else [])
    for li, layer in enumerate(encoder_layers):
        _collect(layer, li)

    model.zero_grad()
    model.eval()
    return bias_scores, perf_scores


def fasp_prune(model, tok, sample_texts, sparsity=0.30, gamma=0.50, device='cpu'):
    bias_scores, perf_scores = compute_head_scores(model, tok, sample_texts, device)
    n_heads = len(bias_scores)
    if n_heads == 0:
        return model, []

    n_protected  = int(n_heads * gamma)
    perf_arr     = np.array(perf_scores)
    perf_thresh  = np.sort(perf_arr)[::-1][n_protected - 1] if n_protected > 0 else np.inf
    protected    = set(i for i, s in enumerate(perf_scores) if s >= perf_thresh)

    candidate_idx   = [i for i in range(n_heads) if i not in protected]
    n_prune         = int(n_heads * sparsity)
    bias_candidates = [(bias_scores[i], i) for i in candidate_idx]
    bias_candidates.sort(reverse=True)
    heads_to_prune  = set(idx for _, idx in bias_candidates[:n_prune])

    head_counter = 0
    encoder_layers = (model.bert.encoder.layer if hasattr(model, 'bert')
                      else model.distilbert.transformer.layer if hasattr(model, 'distilbert')
                      else model.mobilebert.encoder.layer if hasattr(model, 'mobilebert')
                      else [])
    with torch.no_grad():
        for layer in encoder_layers:
            try:
                attn = (layer.attention.self
                        if hasattr(layer, 'attention') and hasattr(layer.attention, 'self')
                        else layer.attention)
                num_heads = attn.num_attention_heads
                head_dim  = attn.attention_head_size if hasattr(attn, 'attention_head_size') \
                            else attn.query.weight.shape[0] // num_heads
                for h in range(num_heads):
                    if head_counter in heads_to_prune:
                        s, e = h * head_dim, (h + 1) * head_dim
                        attn.value.weight[s:e, :] = 0.0
                        if attn.value.bias is not None:
                            attn.value.bias[s:e] = 0.0
                    head_counter += 1
            except Exception:
                pass

    return model, list(heads_to_prune)

In [5]:
def get_model(hf_id, num_labels):
    tok   = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForSequenceClassification.from_pretrained(
        hf_id, num_labels=num_labels, ignore_mismatched_sizes=True
    ).to(DEVICE)
    return tok, model


def memory_mb(model):
    with tempfile.NamedTemporaryFile(delete=False) as f:
        torch.save(model.state_dict(), f.name)
        size_mb = os.path.getsize(f.name) / (1024 * 1024)
    os.remove(f.name)
    return round(size_mb, 2)


def tokenize(tok, texts):
    if isinstance(texts[0], tuple):
        return tok([t[0] for t in texts], [t[1] for t in texts],
                   truncation=True, padding=True,
                   max_length=MAX_LENGTH, return_tensors='pt')
    return tok(texts, truncation=True, padding=True,
               max_length=MAX_LENGTH, return_tensors='pt')


def benchmark(model, tok, ds_cfg):
    path, config, split, text_col, label_col = ds_cfg
    ds = load_dataset(path, config, split=split) if config else load_dataset(path, split=split)
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))

    t_start = time.perf_counter()
    model.eval()
    preds, labels, latencies, energies = [], [], [], []

    for i in range(0, len(ds), BATCH_SIZE):
        batch  = ds[i:i + BATCH_SIZE]
        texts  = (list(zip(batch[text_col[0]], batch[text_col[1]]))
                  if isinstance(text_col, tuple) else batch[text_col])
        first_param  = next(model.parameters(), None)
        model_device = first_param.device if first_param is not None else torch.device('cpu')
        enc = {k: v.to(model_device) for k, v in tokenize(tok, texts).items()}

        sampler = PowerSampler()
        sampler.start()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(**enc)
        if model_device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed_ms   = (time.perf_counter() - t0) * 1000
        avg_power_mw = sampler.stop()

        batch_n   = len(batch[label_col])
        energy_mj = (avg_power_mw * elapsed_ms * 1e-3) / batch_n
        latencies.append(elapsed_ms / batch_n)
        energies.append(energy_mj)

        if config == 'stsb':
            preds.extend(out.logits.squeeze(-1).cpu().tolist())
        else:
            preds.extend(out.logits.argmax(-1).cpu().tolist())
        labels.extend(batch[label_col])

    total_time = time.perf_counter() - t_start
    latency    = np.mean(latencies)
    throughput = len(ds) / total_time
    energy     = np.mean(energies)

    if config == 'stsb':
        pearson  = pearsonr(preds, labels)[0]
        spearman = spearmanr(preds, labels)[0]
        return {'Accuracy': np.nan, 'Precision': np.nan, 'Recall': np.nan, 'F1': np.nan,
                'Pearson': round(pearson * 100, 2), 'Spearman': round(spearman * 100, 2),
                'Latency': round(latency, 4), 'Throughput': round(throughput, 1),
                'Energy': round(energy, 4)}

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average='weighted', zero_division=0)
    rec  = recall_score(labels, preds, average='weighted', zero_division=0)
    f1   = f1_score(labels, preds, average='weighted', zero_division=0)
    return {'Accuracy': round(acc * 100, 2), 'Precision': round(prec * 100, 2),
            'Recall': round(rec * 100, 2), 'F1': round(f1 * 100, 2),
            'Pearson': np.nan, 'Spearman': np.nan,
            'Latency': round(latency, 4), 'Throughput': round(throughput, 1),
            'Energy': round(energy, 4)}

In [ ]:
# Sample texts used to score heads before pruning
SCORE_TEXTS = [
    'This movie is great!',
    'I really disliked this experience.',
    'The weather is nice today.',
    'Science and technology are advancing rapidly.',
    'People from all backgrounds deserve equal treatment.',
] * 4

results = []

for ds_name, ds_cfg in DATASETS.items():
    for model_name, hf_id in MODELS.items():
        print(f'{ds_name} | {model_name}', end=' ... ')
        try:
            tok, model = get_model(hf_id, NUM_LABELS[ds_name])

            model, pruned_heads = fasp_prune(
                model, tok, SCORE_TEXTS,
                sparsity=SPARSITY, gamma=GAMMA, device=DEVICE
            )

            mem     = memory_mb(model)
            metrics = benchmark(model, tok, ds_cfg)

            results.append({
                'Dataset':           ds_name,
                'Model':             model_name,
                'Method':            'FASP',
                'Sparsity (%)':      int(SPARSITY * 100),
                'Heads Pruned':      len(pruned_heads),
                'Memory (MB)':       mem,
                'Latency (ms)':      metrics['Latency'],
                'Throughput (sps)':  metrics['Throughput'],
                'Energy (mJ)':       metrics['Energy'],
                'Accuracy (%)':      metrics['Accuracy'],
                'Precision (%)':     metrics['Precision'],
                'Recall (%)':        metrics['Recall'],
                'F1 (%)':            metrics['F1'],
                'Pearson (%)':       metrics['Pearson'],
                'Spearman (%)':      metrics['Spearman'],
            })
            print('Done')

        except Exception as e:
            print('ERROR:', e)
        finally:
            try:
                del model, tok
            except Exception:
                pass
            gc.collect()
            torch.cuda.empty_cache()

In [7]:
df = pd.DataFrame(results).set_index(['Dataset', 'Model'])
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', '{:.4f}'.format)

for ds in df.index.get_level_values('Dataset').unique():
    print(f'\n========== {ds} ==========')
    print(df.loc[ds].to_string())
print()


========== SST2 ==========
           Method  Sparsity (%)  Heads Pruned  Memory (MB)  Latency (ms)  Throughput (sps)  Energy (mJ)  Accuracy (%)  Precision (%)  Recall (%)  F1 (%)  Pearson (%)  Spearman (%)
Model                                                                                                                                                                              
TinyBERT     FASP            30            14      54.7700        0.3506         1300.3000      12.0167       53.0000        28.0900     53.0000 36.7200          NaN           NaN
DistilBERT   FASP            30             0     255.4500        1.2341          604.1000      70.5976       47.2000        41.8500     47.2000 40.4700          NaN           NaN
AlBERT       FASP            30             0      44.5900        3.1259          269.1000     188.3648       47.2000        51.5600     47.2000 31.8400          NaN           NaN
MobileBERT   FASP            30            28      94.2300        2.9159